# Notebook 03 - Social optimum

This notebook uses the same graph, O-D demands, and candidate route set as Notebook 02. It computes the route-flow assignment that minimizes total system cost over that restricted feasible set.

For edge flows $x$, total system cost is

$$\mathrm{TC}(x)=\sum_{e\in E}x_e t_e(x_e).$$

This objective is different from the Beckmann objective used for user equilibrium. The computation is a system-optimum assignment for the constructed instance; it is not a traffic-control prescription for the real network.

Assumptions:
- The feasible route set is the one constructed in Notebook 02.
- The O-D demands are the same synthetic demands used for UE.
- The BPR cost function and capacity assumptions are unchanged.


In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import pandas as pd
import cvxpy as cp
import osmnx as ox
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

from src.graph_utils import load_graph, bpr

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

print('Libraries loaded.')

## 1. Load UE results

The notebook loads the graph instance, route set, O-D demands, and UE solution saved by Notebook 02.


In [ ]:
with open(PROCESSED_DIR / 'results_ue.pkl', 'rb') as fh:
    ue = pickle.load(fh)

x_ue       = ue['x_ue']
t_ue       = ue['t_ue']
t0         = ue['t0']
cap        = ue['cap']
cost_ue    = ue['cost_ue']
edges_list = ue['edges_list']
od_pairs   = ue['od_pairs']
route_list = ue['route_list']
od_route_map = ue['od_route_map']
delta      = ue['delta']

E = len(edges_list)
R = len(route_list)

print(f'Edges: {E:,}  |  Routes: {R}  |  O-D pairs: {len(od_pairs)}')
print(f'Loaded UE cost: {cost_ue:,.0f} veh·s')

In [ ]:
G = load_graph(RAW_DIR / 'chapinero_drive_enriched.graphml')

## 2. Social-optimum objective in CVXPY

With BPR costs,

$$x_e t_e(x_e)=t_e^0x_e+0.15\,t_e^0\frac{x_e^5}{c_e^4}.$$

For $x_e\ge 0$, $t_e^0>0$, and $c_e>0$, this term is convex. The constraints are the same route-flow demand constraints used in the UE problem.


In [ ]:
def so_objective(x, t0, cap, alpha=0.15, beta=4):
    """
    Total system cost: sum_a x_a * t_a(x_a)
    = sum_a t0_a * (x_a + alpha/(beta+1) * x_a^(beta+1) / cap_a^beta)
    
    OJO: x*t(x) = t0*x + t0*alpha*x^(beta+1)/cap^beta
         (no es la integral de Beckmann, es el producto directo)
    """
    term_linear = cp.sum(cp.multiply(t0, x))
    term_power  = cp.sum(
        cp.multiply(t0 * alpha / (cap ** beta),
                    cp.power(x, beta + 1))
    )
    return term_linear + term_power


# Variables de flow por route
f_so = cp.Variable(R, nonneg=True, name='flow_routes_so')
x_so_expr = delta @ f_so   # edge flows (CVXPY expression)

# Demand-conservation constraints (identical to UE)
constraints_so = []
for (o, d, dem) in od_pairs:
    od_key = (o, d)
    if od_key in od_route_map:
        constraints_so.append(cp.sum(f_so[od_route_map[od_key]]) == dem)

objective_so = cp.Minimize(so_objective(x_so_expr, t0, cap))
problem_so   = cp.Problem(objective_so, constraints_so)

print(f'Problema SO:')
print(f'  Variables     : {R}')
print(f'  Constraints : {len(constraints_so)}')
print(f'  Is DCP?      : {problem_so.is_dcp()}')

## 3. Solve the convex problem

The solution minimizes total system cost over the finite candidate route set. Solver status should be checked before interpreting the numbers.


In [ ]:
problem_so.solve(solver=cp.SCS, eps=1e-4, max_iters=10000, verbose=False)

print(f'Status       : {problem_so.status}')
print(f'Optimal value : {problem_so.value:.4f}')

if problem_so.status not in ['optimal', 'optimal_inaccurate']:
    raise RuntimeError('The solver did not find an optimal solution for SO.')

x_so  = delta @ f_so.value
t_so  = np.array([bpr(t0[i], x_so[i], cap[i]) for i in range(E)])
cost_so = float(np.sum(t_so * x_so))

total_demand = sum(dem for _, _, dem in od_pairs)
avg_time_so  = cost_so / total_demand

print(f'\nSO flow statistics:')
print(f'  Edges with flow > 0 : {(x_so > 1).sum()}')
print(f'  Maximum flow          : {x_so.max():.1f} veh/h')
print(f'  Mean flow (actives) : {x_so[x_so > 1].mean():.1f} veh/h')
print(f'\nTotal cost SO         : {cost_so:,.0f} veh·s')
print(f'Average time SO     : {avg_time_so:.1f} s ({avg_time_so/60:.2f} min)')

## 4. Compare UE and SO objective values

The comparison uses the same total system cost functional for both assignments.


In [ ]:
avg_time_ue = cost_ue / total_demand
poa = cost_ue / cost_so
ineficiencia = (cost_ue - cost_so) / cost_so * 100

print('=' * 50)
print('   UE VS SO COMPARISON')
print('=' * 50)
print(f'  Total cost UE  : {cost_ue:>12,.0f} veh·s')
print(f'  Total cost SO  : {cost_so:>12,.0f} veh·s')
print(f'  Diferencia      : {cost_ue - cost_so:>12,.0f} veh·s  (+{ineficiencia:.2f}%)')
print('-' * 50)
print(f'  Tiempo prom. UE : {avg_time_ue:>8.1f} s  ({avg_time_ue/60:.2f} min)')
print(f'  Tiempo prom. SO : {avg_time_so:>8.1f} s  ({avg_time_so/60:.2f} min)')
print('-' * 50)
print(f'  Price of Anarchy: {poa:.4f}')
print('=' * 50)

## 5. Compare edge-flow changes

The difference $x^{SO}-x^{UE}$ indicates which edges receive more or less flow under the system-optimum assignment for this instance.


In [ ]:
diff_flow = x_so - x_ue   # positive: SO assigns more flow; negative: SO releases flow

df_edges = pd.DataFrame({
    'u': [u for u, v, k in edges_list],
    'v': [v for u, v, k in edges_list],
    'x_ue':   x_ue,
    'x_so':   x_so,
    't0':     t0,
    't_ue':   t_ue,
    't_so':   t_so,
    'cap':    cap,
    'diff':   diff_flow,
    'vcr_ue': x_ue / cap,   # volumen/capacity UE
    'vcr_so': x_so / cap,   # volumen/capacity SO
})

print('Top 10 edges donde SO AGREGA flow (vs UE):')
display(df_edges.nlargest(10, 'diff')[['u','v','x_ue','x_so','diff','cap','vcr_ue','vcr_so']].round(1))

print('\nTop 10 edges donde SO LIBERA flow (vs UE):')
display(df_edges.nsmallest(10, 'diff')[['u','v','x_ue','x_so','diff','cap','vcr_ue','vcr_so']].round(1))

In [ ]:
# Volume-capacity ratio distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, vcr, label, color in [
    (axes[0], df_edges['vcr_ue'], 'Equilibrio de Wardrop (UE)', '#e74c3c'),
    (axes[1], df_edges['vcr_so'], 'Social optimum (SO)',          '#2ecc71'),
]:
    active = vcr[vcr > 0.001]
    ax.hist(active, bins=30, color=color, edgecolor='white', linewidth=0.4)
    ax.axvline(1.0, color='black', linestyle='--', linewidth=1.2, label='Capacidad')
    ax.set_xlabel('Volume/capacity ratio (V/C)', fontsize=11)
    ax.set_ylabel('Number of edges', fontsize=11)
    ax.set_title(label)
    ax.legend()

plt.suptitle('V/C distribution: UE vs SO - Chapinero', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'so_vc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Map edge-flow differences

The map is a diagnostic visualization of $x^{SO}-x^{UE}$. It is not an empirical estimate of changes in observed traffic.


In [ ]:
# Assign difference to the graph
for i, (u, v, k) in enumerate(edges_list):
    G[u][v][k]['flow_so']   = float(x_so[i])
    G[u][v][k]['flow_diff'] = float(diff_flow[i])

# Diverging colormap: red = SO loads more, blue = SO releases
max_diff = np.abs(diff_flow).max()
norm = mcolors.TwoSlopeNorm(vmin=-max_diff, vcenter=0, vmax=max_diff)
cmap = plt.get_cmap('RdBu_r')

edge_colors = [cmap(norm(diff_flow[i])) for i in range(E)]
edge_widths = [0.5 + abs(diff_flow[i]) / 150 for i in range(E)]

fig, ax = ox.plot_graph(
    G,
    edge_color=edge_colors,
    edge_linewidth=edge_widths,
    node_size=5,
    node_color='white',
    bgcolor='#1a1a2e',
    figsize=(12, 10),
    show=False,
    close=False,
)
ax.set_title('Flow difference SO − UE (red: SO loads more, blue: SO releases)\nChapinero, Bogota',
             color='white', fontsize=13, pad=12)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, orientation='vertical', shrink=0.6, label='Δ flow (veh/h)')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'so_flow_diff.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Marginal social cost and external cost

For an edge cost function $t_e$, the marginal social cost is

$$MC_e(x_e)=\frac{d}{dx_e}\bigl[x_e t_e(x_e)\bigr]=t_e(x_e)+x_e t_e'(x_e).$$

For BPR costs with exponent $4$,

$$MC_e(x_e)=t_e^0\left(1+0.15\cdot 5\left(\frac{x_e}{c_e}\right)^4\right).$$

The difference $MC_e(x_e)-t_e(x_e)=x_e t_e'(x_e)$ is the congestion externality on that edge.


In [ ]:
def marginal_cost_bpr(t0, x, cap, alpha=0.15, beta=4):
    """Marginal social cost: t(x) + x*t'(x)"""
    t_x   = t0 * (1 + alpha * (x / cap) ** beta)
    dt_dx = t0 * alpha * beta * (x ** (beta - 1)) / (cap ** beta)
    return t_x + x * dt_dx

mc_so = marginal_cost_bpr(t0, x_so, cap)
pigou_toll = mc_so - t_so   # externality = optimal Pigou toll (seconds)

active = x_so > 1
print('Congestion externality on active edges (Pigou toll, seconds):')
print(f'  Media   : {pigou_toll[active].mean():.1f} s')
print(f'  Maximum  : {pigou_toll[active].max():.1f} s')
print(f'  Median : {np.meann(pigou_toll[active]):.1f} s')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pigou_toll[active], bins=30, color='#9b59b6', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Optimal Pigou toll (seconds)', fontsize=11)
ax.set_ylabel('Number of edges', fontsize=11)
ax.set_title('Pigou toll distribution - Chapinero')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'pigou_toll.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save intermediate results

The saved object contains the UE quantities, SO quantities, and comparison values used by later notebooks.

Limitations of this notebook:
- The SO solution is restricted to the same candidate route set as UE.
- The result depends on synthetic demand and heuristic capacity values.
- The computation does not represent a deployable traffic-management plan.


In [ ]:
results_so = {
    **ue,                         # hereda todo lo de UE
    'x_so':       x_so,
    't_so':       t_so,
    'cost_so':    cost_so,
    'poa':        poa,
    'pigou_toll': pigou_toll,
    'diff_flow':  diff_flow,
}

out_path = PROCESSED_DIR / 'results_so.pkl'
with open(out_path, 'wb') as fh:
    pickle.dump(results_so, fh)

print(f'SO results saved to: {out_path}')
print(f'\nFinal summary:')
print(f'  Costo UE        : {cost_ue:,.0f} veh·s')
print(f'  Costo SO        : {cost_so:,.0f} veh·s')
print(f'  Ahorro SO vs UE : {cost_ue - cost_so:,.0f} veh·s ({ineficiencia:.2f}%)')
print(f'  Price of Anarchy: {poa:.4f}')
print(f'\nReady for Notebook 04 - Price of Anarchy')